# `06_z_sensitivity.ipynb` — v0.2.6 §0.6 CORRECTIONS-Z sensitivity arms

**Spec citation:** `docs/specs/2026-05-16-ai-cost-factor-model-design.md` v0.2.6 §0.6 — CORRECTIONS-Z (Z-1 multi-period aggregation, Z-2 backcast bootstrap, Z-2-W winsorized robustness sub-arm, Z-3 R6 escalation gate, BACKLOG-Z-4).

**Headline framing (CORRECTIONS-K diagnostic-only).** Z-arms are *corroborative* sensitivity tests on R5 PRIMARY (Task 12). They do NOT have verdict authority over the R5/R4-S3 framework. R5 PRIMARY's headline FX share on the daily 2026-Q1-Q2 real data stays the headline. Z-1 tests horizon robustness, Z-2 tests regime/sample-extension robustness, Z-3 routes escalation to the R6 sibling iteration if either fails.

**Anti-fishing posture (mandatory).** ALL pins (seeds, B counts, block lengths, escalation thresholds, kurtosis trigger, backcast horizon) are declared PRE-DATA in Cell 2 and do NOT change based on the values they produce. This notebook is single-pass.

## 9-trio table of contents (CR Q6 pin — locked at the top BEFORE any data cell)

| # | Trio | Frequency / pool | Pinned seed | Pinned B | Pinned block_len |
|---|---|---|---|---|---|
| Cell 3 | Load panel + observed Δln cost^USD + kurtosis | daily, T=28 | — | — | — |
| Z-1a | Weekly aggregation (ISO week, ≥1-obs included) | weekly, T~9 | 20260517 | 10,000 | ⌈T_w^(1/3)⌉ (clamped ≥2) |
| Z-1b | Monthly aggregation (calendar month, ≥1-obs included) | monthly, T~4 | 20260518 | 10,000 | ⌈T_m^(1/3)⌉ (clamped ≥2) — **TAGGED uninformative if T<5; EXCLUDED from Z-3** |
| Z-2-prep | Backcast horizon construction + cost-pool definition | TRM-available dates in [2024-01-03, 2025-12-31] | — | — | — |
| Z-2-main | Backcast bootstrap of Δln cost^USD onto 2024-2025 TRM | bootstrap pool: observed dln_usd | 20260519 | B_PATHS=1,000 | 4 (= daily R5 pin) |
| Z-2-null | Null-calibration: same loop with 2026-Q1-Q2 TRM only | bootstrap pool: observed dln_usd | 20260520 | B_PATHS=1,000 | 4 |
| Z-2-W | Winsorized robustness (CONDITIONAL on observed_kurtosis > 3.0) | bootstrap pool: Winsorized dln_usd at [5%, 95%] | 20260521 | B_PATHS=1,000 | 4 |
| Z-3 | Escalation gate evaluation + close-out routing | aggregator over Z-2-main + Z-2-W (Z-2-null is suppressor) | — | — | — |
| Cell 14 | Headline output + compound close-out (Y-9 + Z-3) | summary table | — | — | — |

## Anti-fishing pins (mirrored in Cell 2; declared here for human review BEFORE Cell 2 executes)

- `DAILY_BASELINE_FX_SHARE = 0.0000277` — from Task 12 R5 (28-day daily).
- `B = 10_000` — within-frequency stationary-bootstrap reps (Z-1a, Z-1b).
- `B_PATHS = 1_000` — backcast paths (Z-2-main, Z-2-null, Z-2-W).
- `SEED_Z1_WEEKLY = 20260517`, `SEED_Z1_MONTHLY = 20260518`.
- `SEED_Z2_MAIN = 20260519`, `SEED_Z2_NULL = 20260520`, `SEED_Z2_W = 20260521`.
- `BACKCAST_START = date(2024, 1, 3)`, `BACKCAST_END = date(2025, 12, 31)`.
- `ESCALATION_REL = 5.0` — escalate if median ≥ 5× daily baseline.
- `ESCALATION_ABS = 0.05` — escalate if median ≥ 0.05 in absolute share.
- `KURTOSIS_TRIGGER = 3.0` — excess kurtosis above which Z-2-W activates.
- Z-1 1-observation bucket rule (CR Q3 pin): INCLUDE 1-obs buckets as 1-day aggregates.
- Z-1 aggregator: SUM (cost, tokens), MEAN (TRM) — CR Q2 pin, conservative against FX-share inflation.
- Z-2 no-reanchoring rule (CR Q4 pin): cost^USD path is `cost_0 · exp(cumsum(dln))` with no periodic level resets.
- Z-3 OR semantics (CR Q5 pin): escalate if median ≥ 5×baseline OR ≥ 0.05 absolute, on EITHER Z-2-main OR Z-2-W.
- Z-3 null suppressor (RC Q5 pin): if Z-2-null median itself trips the 5×baseline gate, Z-3 escalation is SUPPRESSED (gate mis-calibrated).
- Compound close-out gate (CR Q6 pin): close iff BOTH (Z-3 not triggered) AND (Y-9 ccusage-parity-0.1% closed). Y-9 is NOT closed in v0.2.6 → expected outcome is PAUSED, not CLOSED.

## Deviation notice (logged ahead of execution)

The implementer plan stated `BACKCAST_END = 2025-12-31 weekdays only` and added a no-missing-TRM assertion under the claim *"Banrep TRM has only weekdays so this should align"*. Empirical fact-check (pre-execution): Banrep TRM is a CALENDAR-DATE series with Saturdays present and Colombian holidays missing — over [2024-01-03, 2025-12-31] there are 521 weekdays vs 474 TRM rows (148 missing weekdays, 101 non-weekday TRM rows). The spec text in §0.6 reads *"real Banrep daily TRM (already in `data/raw/banrep_trm_daily.parquet`)"*, which is the canonical input. I therefore use **the TRM-available dates in [BACKCAST_START, BACKCAST_END] as the backcast horizon** (N_bc = 474) rather than a synthetic weekday calendar. This is faithful to the spec's canonical TRM source and avoids fabricating TRM values; it's flagged here so the disposition reviewer can see the deviation.

In [1]:
# ============================================================================
# CELL 2 — Anti-fishing pins (DECLARED PRE-DATA, DO NOT TUNE)
# ============================================================================
from __future__ import annotations

import math
from datetime import date
from pathlib import Path

import numpy as np
import polars as pl
from arch.bootstrap import StationaryBootstrap
from scipy.stats import kurtosis

# --- Pinned from Task 12 R5 (daily baseline) --------------------------------
DAILY_BASELINE_FX_SHARE = 0.0000277

# --- Bootstrap repetition counts --------------------------------------------
B = 10_000          # within-frequency bootstrap reps (Z-1a, Z-1b)
B_PATHS = 1_000     # backcast paths (Z-2-main, Z-2-null, Z-2-W)

# --- Seeds (one per arm) ----------------------------------------------------
SEED_Z1_WEEKLY  = 20260517
SEED_Z1_MONTHLY = 20260518
SEED_Z2_MAIN    = 20260519
SEED_Z2_NULL    = 20260520
SEED_Z2_W       = 20260521

# --- Backcast horizon -------------------------------------------------------
BACKCAST_START = date(2024, 1, 3)
BACKCAST_END   = date(2025, 12, 31)

# --- Escalation thresholds (Z-3) --------------------------------------------
ESCALATION_REL    = 5.0      # × daily baseline
ESCALATION_ABS    = 0.05     # absolute FX share
KURTOSIS_TRIGGER  = 3.0      # excess kurtosis → activate Z-2-W

# --- Bootstrap mechanics ----------------------------------------------------
BLOCK_LEN_BACKCAST = 4       # same as daily R5
CI_SIZE = 0.90
CI_METHOD = 'basic'

# --- Paths ------------------------------------------------------------------
PANEL_PATH = Path('../../data/panels/notional_cost_panel.parquet')
TRM_PATH   = Path('../../data/raw/banrep_trm_daily.parquet')

print('=' * 78)
print('ANTI-FISHING PINS — v0.2.6 §0.6 CORRECTIONS-Z (declared PRE-DATA)')
print('=' * 78)
print(f'DAILY_BASELINE_FX_SHARE = {DAILY_BASELINE_FX_SHARE}')
print(f'B (within-freq reps)    = {B}')
print(f'B_PATHS (backcast)      = {B_PATHS}')
print(f'SEED_Z1_WEEKLY={SEED_Z1_WEEKLY}  SEED_Z1_MONTHLY={SEED_Z1_MONTHLY}')
print(f'SEED_Z2_MAIN={SEED_Z2_MAIN}  SEED_Z2_NULL={SEED_Z2_NULL}  SEED_Z2_W={SEED_Z2_W}')
print(f'BACKCAST_START={BACKCAST_START}  BACKCAST_END={BACKCAST_END}')
print(f'ESCALATION_REL={ESCALATION_REL}  ESCALATION_ABS={ESCALATION_ABS}')
print(f'KURTOSIS_TRIGGER={KURTOSIS_TRIGGER}')
print(f'BLOCK_LEN_BACKCAST={BLOCK_LEN_BACKCAST}  CI_SIZE={CI_SIZE}  CI_METHOD={CI_METHOD!r}')
print('=' * 78)

ANTI-FISHING PINS — v0.2.6 §0.6 CORRECTIONS-Z (declared PRE-DATA)
DAILY_BASELINE_FX_SHARE = 2.77e-05
B (within-freq reps)    = 10000
B_PATHS (backcast)      = 1000
SEED_Z1_WEEKLY=20260517  SEED_Z1_MONTHLY=20260518
SEED_Z2_MAIN=20260519  SEED_Z2_NULL=20260520  SEED_Z2_W=20260521
BACKCAST_START=2024-01-03  BACKCAST_END=2025-12-31
ESCALATION_REL=5.0  ESCALATION_ABS=0.05
KURTOSIS_TRIGGER=3.0
BLOCK_LEN_BACKCAST=4  CI_SIZE=0.9  CI_METHOD='basic'


## Cell 3 — Load production panel + observed Δln cost^USD + sample excess kurtosis

Reads the production panel (29 daily rows, ccusage-parity 0.9994×), computes the 28-element `dln_usd = Δln notional_cost_usd` series that will be the bootstrap pool for Z-2, and computes the sample excess kurtosis (Fisher, unbiased) that gates Z-2-W activation.

This cell also establishes `COST_START_USD = median(observed cost_usd)` as the backcast starting level (CR Q4 pin: no re-anchoring downstream).

In [2]:
df = pl.read_parquet(PANEL_PATH)
print(f'panel shape:        {df.shape}')
print(f'panel date window:  {df["date_utc"].min()} -> {df["date_utc"].max()}')

cost_usd_obs = df['notional_cost_usd'].to_numpy()
cost_cop_obs = df['notional_cost_cop'].to_numpy()
trm_obs      = df['trm_cop_per_usd'].to_numpy()

dln_usd = np.diff(np.log(cost_usd_obs))
dln_cop = np.diff(np.log(cost_cop_obs))
dln_trm = np.diff(np.log(trm_obs))
T_daily = len(dln_usd)

print(f'\nT_daily (after first-diff) = {T_daily}')
print(f'dln_usd stats: mean={dln_usd.mean():+.5f}, sd={dln_usd.std(ddof=1):.5f}')

# Sample excess kurtosis (Fisher, unbiased) — gates Z-2-W
observed_kurtosis = float(kurtosis(dln_usd, fisher=True, bias=False))
print(f'\nobserved_kurtosis (Fisher excess, unbiased) = {observed_kurtosis:.4f}')
if observed_kurtosis > KURTOSIS_TRIGGER:
    print(f'  -> > {KURTOSIS_TRIGGER}  =>  Z-2-W WILL RUN (heavy-tail diagnostic activated)')
else:
    print(f'  -> <= {KURTOSIS_TRIGGER}  =>  Z-2-W WILL BE SKIPPED')

# Backcast starting level (no-re-anchoring pin: this is used ONCE at t=0)
COST_START_USD = float(np.median(cost_usd_obs))
print(f'\nCOST_START_USD (median observed cost_usd) = {COST_START_USD:.4f}')

# Sanity: replicate daily baseline ≈ 0.0000277 from the panel itself
fx_share_daily_check = float(np.var(dln_trm) / np.var(dln_cop))
print(f'\nSanity — daily FX share from panel (Var(Δln TRM)/Var(Δln Cost^COP)) = {fx_share_daily_check:.7f}')
print(f'         pinned baseline                                              = {DAILY_BASELINE_FX_SHARE:.7f}')
rel_match = abs(fx_share_daily_check - DAILY_BASELINE_FX_SHARE) / DAILY_BASELINE_FX_SHARE
print(f'         relative match to pin                                        = {rel_match:.2%}')

panel shape:        (29, 11)
panel date window:  2026-01-06 -> 2026-05-14

T_daily (after first-diff) = 28
dln_usd stats: mean=+0.02014, sd=1.65642

observed_kurtosis (Fisher excess, unbiased) = 3.6237
  -> > 3.0  =>  Z-2-W WILL RUN (heavy-tail diagnostic activated)

COST_START_USD (median observed cost_usd) = 72.3050

Sanity — daily FX share from panel (Var(Δln TRM)/Var(Δln Cost^COP)) = 0.0000283
         pinned baseline                                              = 0.0000277
         relative match to pin                                        = 2.27%


## Trio Z-1a — Weekly aggregation

### Decision-citation block (4-part)

- **Reference:** spec §0.6 CORRECTIONS-Z-1 (weekly row of the multi-period table).
- **Why:** test horizon robustness of the small-FX finding. R5 PRIMARY's daily reading could in principle be diluted by within-week intra-day noise; weekly aggregation collapses that noise.
- **Relevance:** stationary bootstrap on the ~9-bucket weekly Δln series, with the same `fx_share_excl_cov` / `usage_share_excl_cov` / `two_cov` functions as Task 12 R5 (CORRECTIONS-Q conservative cov rule carried).
- **Connection:** weekly FX share + 90% CI feed Z-3 alongside the daily baseline. Per the spec's pre-pinned interpretation rule, weekly within ±2× daily ⇒ small-FX is NOT a horizon artifact; weekly > 5× daily ⇒ daily reading diluted.

### Why

Aggregate the 28-day panel into ISO weeks (Year+Week) per the spec: sum of cost / tokens, simple-mean of TRM. The 1-obs-bucket rule (CR Q3 pin) keeps 1-day weeks as 1-day aggregates — excluding them would let the implementer change N post-hoc.

In [3]:
# ---- Aggregator functions (carry from Task 12 R5; same signatures) ---------
def fx_share_excl_cov(c: np.ndarray, u: np.ndarray, x: np.ndarray) -> float:
    """Var(Δln USDCOP) / Var(Δln Cost^COP) — CORRECTIONS-Q conservative."""
    return float(np.var(x) / np.var(c))

def usage_share_excl_cov(c: np.ndarray, u: np.ndarray, x: np.ndarray) -> float:
    """Var(Δln Cost^USD) / Var(Δln Cost^COP) — CORRECTIONS-Q conservative."""
    return float(np.var(u) / np.var(c))

def two_cov(c: np.ndarray, u: np.ndarray, x: np.ndarray) -> float:
    """2·Cov(Δln Cost^USD, Δln USDCOP) — diagnostic, separately reported."""
    return float(2.0 * np.cov(u, x, ddof=0)[0, 1])

# ---- Weekly bucketing ------------------------------------------------------
weekly = (
    df.with_columns(
        iso_year=pl.col('date_utc').dt.iso_year(),
        iso_week=pl.col('date_utc').dt.week(),
    )
    .group_by(['iso_year', 'iso_week'])
    .agg([
        pl.col('notional_cost_usd').sum().alias('cost_usd'),
        pl.col('notional_cost_cop').sum().alias('cost_cop'),
        pl.col('input_tok').sum().alias('input_tok'),
        pl.col('output_tok').sum().alias('output_tok'),
        pl.col('cache_create_5m').sum().alias('cache_create_5m'),
        pl.col('cache_create_1h').sum().alias('cache_create_1h'),
        pl.col('cache_read').sum().alias('cache_read'),
        pl.col('trm_cop_per_usd').mean().alias('trm'),  # CR Q2: simple-mean
        pl.len().alias('n_obs'),
    ])
    .sort(['iso_year', 'iso_week'])
)
print(f'Weekly buckets: T_weekly = {weekly.height} (1-obs buckets INCLUDED per CR Q3 pin)')
print(weekly.select(['iso_year', 'iso_week', 'n_obs', 'cost_usd', 'cost_cop', 'trm']))

T_weekly = weekly.height
wk_cost_usd = weekly['cost_usd'].to_numpy()
wk_cost_cop = weekly['cost_cop'].to_numpy()
wk_trm      = weekly['trm'].to_numpy()

wk_dln_usd = np.diff(np.log(wk_cost_usd))
wk_dln_cop = np.diff(np.log(wk_cost_cop))
wk_dln_trm = np.diff(np.log(wk_trm))
T_wk_dln = len(wk_dln_cop)

BLOCK_LEN_WEEKLY = max(2, math.ceil(T_wk_dln ** (1.0 / 3.0)))
print(f'\nT_weekly Δln length = {T_wk_dln},  block_len = max(2, ceil(T^(1/3))) = {BLOCK_LEN_WEEKLY}')

# ---- Point + 90% CI on weekly FX share -------------------------------------
pt_fx_weekly = fx_share_excl_cov(wk_dln_cop, wk_dln_usd, wk_dln_trm)
pt_us_weekly = usage_share_excl_cov(wk_dln_cop, wk_dln_usd, wk_dln_trm)
pt_cv_weekly = two_cov(wk_dln_cop, wk_dln_usd, wk_dln_trm)

sb_wk = StationaryBootstrap(BLOCK_LEN_WEEKLY, wk_dln_cop, wk_dln_usd, wk_dln_trm, seed=SEED_Z1_WEEKLY)
ci_fx_wk = sb_wk.conf_int(fx_share_excl_cov, reps=B, size=CI_SIZE, method=CI_METHOD)
fx_wk_lo, fx_wk_hi = float(ci_fx_wk[0, 0]), float(ci_fx_wk[1, 0])

print('\n--- Z-1a weekly variance decomposition (CORRECTIONS-Q conservative) ---')
print(f'FX share (weekly)    = {pt_fx_weekly:.6f}   | 90% CI = [{fx_wk_lo:.6f}, {fx_wk_hi:.6f}]')
print(f'Usage share (weekly) = {pt_us_weekly:.6f}   ')
print(f'2·Cov (weekly)       = {pt_cv_weekly:.6f}   ')
print(f'\nDaily baseline FX share (Task 12 R5) = {DAILY_BASELINE_FX_SHARE:.7f}')
rel_to_daily = pt_fx_weekly / DAILY_BASELINE_FX_SHARE if DAILY_BASELINE_FX_SHARE > 0 else float("inf")
print(f'Weekly / Daily ratio = {rel_to_daily:.2f}x   (spec pre-pinned rule: within ±2x = not a horizon artifact)')

Weekly buckets: T_weekly = 9 (1-obs buckets INCLUDED per CR Q3 pin)
shape: (9, 6)
┌──────────┬──────────┬───────┬────────────┬───────────────┬─────────────┐
│ iso_year ┆ iso_week ┆ n_obs ┆ cost_usd   ┆ cost_cop      ┆ trm         │
│ ---      ┆ ---      ┆ ---   ┆ ---        ┆ ---           ┆ ---         │
│ i32      ┆ i8       ┆ u32   ┆ f64        ┆ f64           ┆ f64         │
╞══════════╪══════════╪═══════╪════════════╪═══════════════╪═════════════╡
│ 2026     ┆ 2        ┆ 2     ┆ 0.5745845  ┆ 2154.021946   ┆ 3750.145    │
│ 2026     ┆ 10       ┆ 4     ┆ 114.201898 ┆ 430153.072164 ┆ 3771.43     │
│ 2026     ┆ 11       ┆ 4     ┆ 323.27889  ┆ 1.1998e6      ┆ 3710.9175   │
│ 2026     ┆ 15       ┆ 3     ┆ 167.877995 ┆ 611526.808056 ┆ 3653.916667 │
│ 2026     ┆ 16       ┆ 4     ┆ 456.293032 ┆ 1.6436e6      ┆ 3601.3025   │
│ 2026     ┆ 17       ┆ 2     ┆ 310.240498 ┆ 1.1051e6      ┆ 3564.75     │
│ 2026     ┆ 18       ┆ 3     ┆ 204.041253 ┆ 734323.341472 ┆ 3616.263333 │
│ 2026     ┆ 19   

### Interpretation (Z-1a)

The weekly FX share's relation to the daily baseline (0.0000277) tells us whether the small-FX finding is a horizon artifact: per the spec's pre-pinned rule, within ±2× ⇒ not a horizon artifact; >5× ⇒ daily reading diluted by within-week intra-day noise. The 90% bootstrap CI is wide at T=8 (post-diff) and is intended as a directional reading, not a precise inference.

## Trio Z-1b — Monthly aggregation

### Decision-citation block (4-part)

- **Reference:** spec §0.6 CORRECTIONS-Z-1 (monthly row of the multi-period table).
- **Why:** same horizon-robustness purpose as Z-1a but at calendar-month granularity. Provides a directional sanity check for the operator.
- **Relevance:** EXPLICITLY uninformative for inference — T_monthly=4 is below the bootstrap CI-usability floor. The spec RC Q3 pin REQUIRES tagging this stratum as "uninformative — N=4" and EXCLUDING it from the Z-3 escalation-gate input.
- **Connection:** monthly stratum is REPORTED but does NOT feed Z-3. Carries the same aggregator rule (sum cost/tokens, mean TRM) and 1-obs-bucket inclusion rule as Z-1a.

### Why

Bucket the 28-day panel by calendar (year, month); compute Δln series + FX share + CI; tag as uninformative if T_monthly_diff < 5; EXCLUDE from Z-3.

In [4]:
monthly = (
    df.with_columns(
        year=pl.col('date_utc').dt.year(),
        month=pl.col('date_utc').dt.month(),
    )
    .group_by(['year', 'month'])
    .agg([
        pl.col('notional_cost_usd').sum().alias('cost_usd'),
        pl.col('notional_cost_cop').sum().alias('cost_cop'),
        pl.col('input_tok').sum().alias('input_tok'),
        pl.col('output_tok').sum().alias('output_tok'),
        pl.col('cache_create_5m').sum().alias('cache_create_5m'),
        pl.col('cache_create_1h').sum().alias('cache_create_1h'),
        pl.col('cache_read').sum().alias('cache_read'),
        pl.col('trm_cop_per_usd').mean().alias('trm'),
        pl.len().alias('n_obs'),
    ])
    .sort(['year', 'month'])
)
print(f'Monthly buckets: T_monthly = {monthly.height} (1-obs buckets INCLUDED per CR Q3 pin)')
print(monthly.select(['year', 'month', 'n_obs', 'cost_usd', 'cost_cop', 'trm']))

T_monthly = monthly.height
mo_cost_usd = monthly['cost_usd'].to_numpy()
mo_cost_cop = monthly['cost_cop'].to_numpy()
mo_trm      = monthly['trm'].to_numpy()

mo_dln_usd = np.diff(np.log(mo_cost_usd))
mo_dln_cop = np.diff(np.log(mo_cost_cop))
mo_dln_trm = np.diff(np.log(mo_trm))
T_mo_dln = len(mo_dln_cop)

BLOCK_LEN_MONTHLY = max(2, math.ceil(max(T_mo_dln, 1) ** (1.0 / 3.0)))
print(f'\nT_monthly Δln length = {T_mo_dln},  block_len = max(2, ceil(T^(1/3))) = {BLOCK_LEN_MONTHLY}')

pt_fx_monthly = fx_share_excl_cov(mo_dln_cop, mo_dln_usd, mo_dln_trm)
pt_us_monthly = usage_share_excl_cov(mo_dln_cop, mo_dln_usd, mo_dln_trm)
pt_cv_monthly = two_cov(mo_dln_cop, mo_dln_usd, mo_dln_trm)

UNINFORMATIVE_MONTHLY = (T_mo_dln < 5)
if T_mo_dln >= 2:
    sb_mo = StationaryBootstrap(BLOCK_LEN_MONTHLY, mo_dln_cop, mo_dln_usd, mo_dln_trm, seed=SEED_Z1_MONTHLY)
    ci_fx_mo = sb_mo.conf_int(fx_share_excl_cov, reps=B, size=CI_SIZE, method=CI_METHOD)
    fx_mo_lo, fx_mo_hi = float(ci_fx_mo[0, 0]), float(ci_fx_mo[1, 0])
else:
    fx_mo_lo = fx_mo_hi = float('nan')

print('\n--- Z-1b monthly variance decomposition (UNINFORMATIVE — EXCLUDED from Z-3) ---')
print(f'FX share (monthly)    = {pt_fx_monthly:.6f}   | 90% CI = [{fx_mo_lo:.6f}, {fx_mo_hi:.6f}]')
print(f'Usage share (monthly) = {pt_us_monthly:.6f}')
print(f'2·Cov (monthly)       = {pt_cv_monthly:.6f}')
print(f'\nTAG: uninformative — T_monthly_diff={T_mo_dln} < 5 (below bootstrap usability floor). '
      f'EXCLUDED from Z-3 input per spec RC Q3 pin.')

Monthly buckets: T_monthly = 4 (1-obs buckets INCLUDED per CR Q3 pin)
shape: (4, 6)
┌──────┬───────┬───────┬─────────────┬─────────────┬─────────────┐
│ year ┆ month ┆ n_obs ┆ cost_usd    ┆ cost_cop    ┆ trm         │
│ ---  ┆ ---   ┆ ---   ┆ ---         ┆ ---         ┆ ---         │
│ i32  ┆ i8    ┆ u32   ┆ f64         ┆ f64         ┆ f64         │
╞══════╪═══════╪═══════╪═════════════╪═════════════╪═════════════╡
│ 2026 ┆ 1     ┆ 2     ┆ 0.5745845   ┆ 2154.021946 ┆ 3750.145    │
│ 2026 ┆ 3     ┆ 8     ┆ 437.480788  ┆ 1.6299e6    ┆ 3741.17375  │
│ 2026 ┆ 4     ┆ 12    ┆ 1138.452778 ┆ 4.0945e6    ┆ 3612.104167 │
│ 2026 ┆ 5     ┆ 7     ┆ 1220.020118 ┆ 4.5525e6    ┆ 3742.228571 │
└──────┴───────┴───────┴─────────────┴─────────────┴─────────────┘

T_monthly Δln length = 3,  block_len = max(2, ceil(T^(1/3))) = 2



--- Z-1b monthly variance decomposition (UNINFORMATIVE — EXCLUDED from Z-3) ---
FX share (monthly)    = 0.000099   | 90% CI = [nan, nan]
Usage share (monthly) = 1.004681
2·Cov (monthly)       = -0.027750

TAG: uninformative — T_monthly_diff=3 < 5 (below bootstrap usability floor). EXCLUDED from Z-3 input per spec RC Q3 pin.


/tmp/ipykernel_13855/3995832565.py:4: RuntimeWarning: invalid value encountered in scalar divide
  return float(np.var(x) / np.var(c))


### Interpretation (Z-1b)

Monthly FX share is reported as a directional sanity check ONLY — the T=3 first-difference series is far below the bootstrap CI-usability floor. The CI displayed should be interpreted as nominal: it does not have inferential weight. Per the spec's RC Q3 pin, monthly is EXCLUDED from Z-3.

## Trio Z-2-prep — Backcast horizon construction + cost-pool definition

### Decision-citation block (4-part)

- **Reference:** spec §0.6 CORRECTIONS-Z-2 (reframed per RC FLAG Q4 — drift + extended-sample channels, NOT a high-vol regime; high-vol stress test is BACKLOG-Z-4 for v0.2.7).
- **Why:** test whether the small-FX finding holds when the cost-behavior distribution from 2026-Q1-Q2 is composed with a longer-history (2024-2025) TRM series. The 2024-2025 TRM has larger CUMULATIVE drift (~19% range) than 2026-Q1-Q2 (~7%) but LOWER within-day innovation vol (0.00473 vs 0.00626).
- **Relevance:** holds cost behavior fixed at the observed empirical distribution (the 28-element `dln_usd` series) and swaps in real Banrep TRM for the longer window.
- **Connection:** prepares inputs for Z-2-main, Z-2-null, Z-2-W; outputs N_backcast and the TRM array.

### Why

Build the backcast horizon (TRM-available dates within [BACKCAST_START, BACKCAST_END]; deviation logged in Cell 1 — TRM is calendar-date, not weekday-only). The cost pool is `dln_usd` (length 28) from Cell 3. Output: N_backcast, TRM range/innovation vol, no-missing-TRM assertion against TRM's OWN date set.

In [5]:
trm_full = pl.read_parquet(TRM_PATH).with_columns(
    pl.col('trm_cop_per_usd').cast(pl.Float64)
)
print(f'TRM parquet shape:  {trm_full.shape}')
print(f'TRM date window:    {trm_full["date"].min()} -> {trm_full["date"].max()}')

trm_backcast = (
    trm_full
    .filter((pl.col('date') >= BACKCAST_START) & (pl.col('date') <= BACKCAST_END))
    .sort('date')
)
N_backcast_dates = trm_backcast.height
print(f'\nBackcast horizon (TRM-available dates in [{BACKCAST_START}, {BACKCAST_END}]):')
print(f'  N_backcast = {N_backcast_dates}')
print(f'  date range = {trm_backcast["date"].min()} -> {trm_backcast["date"].max()}')

trm_arr_backcast = trm_backcast['trm_cop_per_usd'].to_numpy()
trm_min, trm_max = float(trm_arr_backcast.min()), float(trm_arr_backcast.max())
trm_rel_range = (trm_max - trm_min) / trm_arr_backcast.mean()
print(f'\nTRM stats over backcast:')
print(f'  min={trm_min:.2f}  max={trm_max:.2f}  mean={trm_arr_backcast.mean():.2f}')
print(f'  relative range (max-min)/mean = {trm_rel_range:.2%}')

dln_trm_backcast = np.diff(np.log(trm_arr_backcast))
trm_innovation_vol_bc = float(dln_trm_backcast.std(ddof=1))
trm_innovation_vol_obs = float(dln_trm.std(ddof=1))
print(f'\nΔln TRM innovation vol:')
print(f'  backcast 2024-01-03..2025-12-31 = {trm_innovation_vol_bc:.5f}')
print(f'  observed 2026-Q1-Q2 window      = {trm_innovation_vol_obs:.5f}')
print(f'  (spec note: backcast vol is LOWER than observed — Z-2 tests drift + sample-extension, NOT high-vol)')

# No-missing-TRM assertion against TRM's own date set (per the deviation logged in Cell 1)
n_trm_unique = trm_backcast['date'].n_unique()
assert n_trm_unique == N_backcast_dates, (
    f'TRM date dedup failed: unique={n_trm_unique} vs rows={N_backcast_dates}'
)
print(f'\nNo-data assertion (TRM date uniqueness within horizon): PASS '
      f'({n_trm_unique} unique dates over {N_backcast_dates} rows)')
print(f'\nCost pool: observed dln_usd, len={T_daily} (used as bootstrap source for Z-2 arms)')
print(f'COST_START_USD = {COST_START_USD:.4f}  (single-use anchor at t=0; no re-anchoring downstream)')

TRM parquet shape:  (563, 2)
TRM date window:    2024-01-03 -> 2026-05-16

Backcast horizon (TRM-available dates in [2024-01-03, 2025-12-31]):
  N_backcast = 474
  date range = 2024-01-03 -> 2025-12-31

TRM stats over backcast:
  min=3706.94  max=4478.21  mean=4063.70
  relative range (max-min)/mean = 18.98%

Δln TRM innovation vol:
  backcast 2024-01-03..2025-12-31 = 0.00713
  observed 2026-Q1-Q2 window      = 0.00881
  (spec note: backcast vol is LOWER than observed — Z-2 tests drift + sample-extension, NOT high-vol)

No-data assertion (TRM date uniqueness within horizon): PASS (474 unique dates over 474 rows)

Cost pool: observed dln_usd, len=28 (used as bootstrap source for Z-2 arms)
COST_START_USD = 72.3050  (single-use anchor at t=0; no re-anchoring downstream)


### Interpretation (Z-2-prep)

The backcast horizon is N_backcast=474 (TRM-available dates in [2024-01-03, 2025-12-31]). The 2024-2025 TRM has LOWER innovation vol than the observed 2026-Q1-Q2 window — confirming the spec's RC Q4 reframe that Z-2 tests sample-extension / drift, not high-vol. The cost pool is the 28-element `dln_usd`; bootstrap will resample it to length N_backcast-1, then reconstruct cost^USD path via `cost_0 · exp(cumsum)` (no re-anchoring, CR Q4 pin).

## Trio Z-2-main — Backcast bootstrap (B_PATHS=1,000 paths against 2024-2025 TRM)

### Decision-citation block (4-part)

- **Reference:** spec §0.6 CORRECTIONS-Z-2 main arm.
- **Why:** the median FX share across paths is the regime-corroboration metric — does the small-FX finding survive when composed with a different TRM regime?
- **Relevance:** stationary bootstrap (block_len=4 = daily R5 pin) preserves within-pool serial structure of `dln_usd`; long-horizon resampling is implemented via geometric block lengths (Politis-Romano).
- **Connection:** feeds Z-3 escalation gate. CR Q4 pin: no re-anchoring along the 474-day path. CR Q5 pin: escalation is OR(rel ≥ 5×, abs ≥ 0.05).

### Why

For each of B_PATHS=1,000 paths: (1) draw a length-(N_backcast-1) `Δln cost^USD` path via stationary bootstrap of the observed 28-element pool; (2) reconstruct `cost^USD_t = COST_START_USD · exp(cumsum(Δln))`; (3) combine with real backcast TRM to get `cost^COP_t = cost^USD_t · TRM_t`; (4) compute the per-path FX share = `Var(Δln TRM)/Var(Δln cost^COP)`. Output: median across paths + 90% inter-path interval.

**Stationary-bootstrap recipe for variable output length.** `arch.bootstrap.StationaryBootstrap.bootstrap()` returns a sample of the SAME length as the input pool. The backcast needs length N_backcast-1 ≫ 28, so we implement the Politis-Romano stationary block draw directly: geometric block lengths with mean = `BLOCK_LEN_BACKCAST`, indices drawn uniformly from the pool, concatenated and truncated to the target length. This is the canonical extension of the stationary bootstrap to a longer output horizon and is mathematically equivalent to the `arch` implementation at the same target length.

In [6]:
def stationary_bootstrap_path(pool: np.ndarray, target_len: int, block_len: int,
                              rng: np.random.Generator) -> np.ndarray:
    """Politis-Romano stationary bootstrap, extended to arbitrary output length.

    Geometric block lengths with mean `block_len`; each block is a contiguous
    slice of `pool` starting at a uniform random index, with circular wrap.
    Concatenated blocks are truncated to `target_len`.
    """
    n_pool = len(pool)
    p = 1.0 / block_len  # geometric success probability so E[block_len] = 1/p
    out = np.empty(target_len, dtype=pool.dtype)
    filled = 0
    while filled < target_len:
        start = int(rng.integers(0, n_pool))
        blen = int(rng.geometric(p))  # ≥ 1
        # circular indexing into pool
        idx = (start + np.arange(blen)) % n_pool
        take = min(blen, target_len - filled)
        out[filled:filled + take] = pool[idx[:take]]
        filled += take
    return out


def backcast_fx_shares(pool_dln_usd: np.ndarray, trm_arr: np.ndarray,
                       cost_start_usd: float, n_paths: int,
                       block_len: int, seed: int) -> np.ndarray:
    """Run B_paths stationary-bootstrap backcasts; return per-path FX share array."""
    rng = np.random.default_rng(seed)
    target_len_dln = len(trm_arr) - 1  # so reconstructed cost^USD has len(trm_arr)
    dln_trm_path = np.diff(np.log(trm_arr))
    var_trm_path = float(np.var(dln_trm_path))
    shares = np.empty(n_paths)
    for i in range(n_paths):
        dln_path = stationary_bootstrap_path(pool_dln_usd, target_len_dln, block_len, rng)
        cost_usd_path = cost_start_usd * np.exp(np.concatenate(([0.0], np.cumsum(dln_path))))
        cost_cop_path = cost_usd_path * trm_arr
        dln_cop_path = np.diff(np.log(cost_cop_path))
        shares[i] = var_trm_path / float(np.var(dln_cop_path))
    return shares


# ---- Z-2-main run ----------------------------------------------------------
fx_shares_main = backcast_fx_shares(
    pool_dln_usd=dln_usd,
    trm_arr=trm_arr_backcast,
    cost_start_usd=COST_START_USD,
    n_paths=B_PATHS,
    block_len=BLOCK_LEN_BACKCAST,
    seed=SEED_Z2_MAIN,
)
median_fx_main = float(np.median(fx_shares_main))
ci_main_lo, ci_main_hi = float(np.quantile(fx_shares_main, 0.05)), float(np.quantile(fx_shares_main, 0.95))

print('=' * 78)
print('Z-2-main — backcast bootstrap (2024-2025 TRM, observed cost-behavior pool)')
print('=' * 78)
print(f'B_PATHS = {B_PATHS}, block_len = {BLOCK_LEN_BACKCAST}, seed = {SEED_Z2_MAIN}')
print(f'N_backcast (TRM dates) = {len(trm_arr_backcast)}')
print(f'\nPer-path FX share distribution:')
print(f'  median             = {median_fx_main:.7f}')
print(f'  90% inter-path CI  = [{ci_main_lo:.7f}, {ci_main_hi:.7f}]')
print(f'  min / max          = [{fx_shares_main.min():.7f}, {fx_shares_main.max():.7f}]')
print(f'\nDaily baseline FX share (Task 12 R5) = {DAILY_BASELINE_FX_SHARE:.7f}')
print(f'Z-2-main median / daily baseline      = {median_fx_main / DAILY_BASELINE_FX_SHARE:.2f}x')
print(f'Pre-pinned escalation rule: ESCALATE if median ≥ {ESCALATION_REL}x baseline ({ESCALATION_REL * DAILY_BASELINE_FX_SHARE:.7f}) '
      f'OR ≥ {ESCALATION_ABS} absolute')

Z-2-main — backcast bootstrap (2024-2025 TRM, observed cost-behavior pool)
B_PATHS = 1000, block_len = 4, seed = 20260519
N_backcast (TRM dates) = 474

Per-path FX share distribution:
  median             = 0.0000194
  90% inter-path CI  = [0.0000169, 0.0000223]
  min / max          = [0.0000148, 0.0000247]

Daily baseline FX share (Task 12 R5) = 0.0000277
Z-2-main median / daily baseline      = 0.70x
Pre-pinned escalation rule: ESCALATE if median ≥ 5.0x baseline (0.0001385) OR ≥ 0.05 absolute


### Interpretation (Z-2-main)

The Z-2-main median FX share is the regime-corroboration headline: if it stays close to the daily baseline (0.0000277), the small-FX finding is robust to sample-extension (it doesn't depend on the particular 28-day 2026-Q1-Q2 TRM window). The 90% inter-path interval reflects path-to-path bootstrap noise. The compound escalation rule is evaluated in Z-3 — Z-2-main alone does not trigger; Z-2-null calibration suppression takes precedence.

## Trio Z-2-null — Null calibration (2026-Q1-Q2 TRM, same machinery)

### Decision-citation block (4-part)

- **Reference:** spec §0.6 CORRECTIONS-Z-2 null-calibration sub-step (RC Q5 pin).
- **Why:** Monte-Carlo noise check. If the same bootstrap loop using the OBSERVED 28-day TRM (not the extended 2024-2025 TRM) itself trips the 5× gate, the gate is mis-calibrated and Z-3 escalation must be suppressed.
- **Relevance:** prevents bootstrap noise from spuriously triggering Z-3. Different seed (20260520) from Z-2-main (20260519) so the null is independent.
- **Connection:** Z-3 evaluates `NULL_GATE_TRIPPED = (null_median ≥ ESCALATION_REL × baseline OR ≥ ESCALATION_ABS)` and SUPPRESSES escalation if so.

### Why

Re-run the Z-2 backcast machinery with `trm_arr = trm_obs` (the panel's own 29-day TRM) instead of the 474-day extended TRM. Median across B_PATHS=1,000 paths should match the daily baseline (0.0000277) within Monte-Carlo noise (≈ same population from which the daily baseline was drawn). Note: with target_len_dln = 28, this is essentially a self-consistency check of the bootstrap mechanics on the original sample.

In [7]:
fx_shares_null = backcast_fx_shares(
    pool_dln_usd=dln_usd,
    trm_arr=trm_obs,                  # observed 29-day TRM (the null)
    cost_start_usd=COST_START_USD,
    n_paths=B_PATHS,
    block_len=BLOCK_LEN_BACKCAST,
    seed=SEED_Z2_NULL,
)
median_fx_null = float(np.median(fx_shares_null))
ci_null_lo, ci_null_hi = float(np.quantile(fx_shares_null, 0.05)), float(np.quantile(fx_shares_null, 0.95))

print('=' * 78)
print('Z-2-null — null calibration (observed 2026-Q1-Q2 TRM, same bootstrap mechanics)')
print('=' * 78)
print(f'B_PATHS = {B_PATHS}, block_len = {BLOCK_LEN_BACKCAST}, seed = {SEED_Z2_NULL}')
print(f'TRM input length (observed window) = {len(trm_obs)}')
print(f'\nPer-path FX share distribution (null):')
print(f'  median             = {median_fx_null:.7f}')
print(f'  90% inter-path CI  = [{ci_null_lo:.7f}, {ci_null_hi:.7f}]')
print(f'\nDaily baseline FX share = {DAILY_BASELINE_FX_SHARE:.7f}')
rel_null_to_baseline = median_fx_null / DAILY_BASELINE_FX_SHARE if DAILY_BASELINE_FX_SHARE > 0 else float('inf')
print(f'Null median / daily baseline = {rel_null_to_baseline:.2f}x')

null_threshold = ESCALATION_REL * DAILY_BASELINE_FX_SHARE
NULL_GATE_TRIPPED = bool((median_fx_null >= null_threshold) or (median_fx_null >= ESCALATION_ABS))
if NULL_GATE_TRIPPED:
    print(f'\n!! NULL CALIBRATION FAILED: null median {median_fx_null:.7f} trips the escalation gate.')
    print(f'   => Z-3 escalation will be SUPPRESSED with a diagnostic memo (gate mis-calibrated).')
else:
    print(f'\nNULL CALIBRATION PASS: null median {median_fx_null:.7f} < {null_threshold:.7f} (5× baseline) '
          f'AND < {ESCALATION_ABS} (absolute).')
    print(f'   => Z-3 may proceed to evaluate Z-2-main / Z-2-W normally.')

Z-2-null — null calibration (observed 2026-Q1-Q2 TRM, same bootstrap mechanics)
B_PATHS = 1000, block_len = 4, seed = 20260520
TRM input length (observed window) = 29

Per-path FX share distribution (null):
  median             = 0.0000293
  90% inter-path CI  = [0.0000182, 0.0000581]

Daily baseline FX share = 0.0000277
Null median / daily baseline = 1.06x

NULL CALIBRATION PASS: null median 0.0000293 < 0.0001385 (5× baseline) AND < 0.05 (absolute).
   => Z-3 may proceed to evaluate Z-2-main / Z-2-W normally.


### Interpretation (Z-2-null)

Null calibration confirms the bootstrap machinery does not by itself drive FX share above the escalation threshold when re-run on the original 28-day TRM input. If this trips, Z-3 is suppressed (the test cannot distinguish signal from MC noise); if it passes, Z-3 evaluates Z-2-main + Z-2-W normally.

## Trio Z-2-W — Winsorized robustness (CONDITIONAL on observed_kurtosis > 3.0)

### Decision-citation block (4-part)

- **Reference:** spec §0.6 CORRECTIONS-Z-2-W (RC Q8 pin).
- **Why:** heavy-tail risk in the 28-day `dln_usd` pool. Winsorizing at [5%, 95%] removes extreme blocks that could either (A) inflate the denominator and hide a real signal, or (B) under-represent outliers in low-burst paths and spuriously inflate FX share.
- **Relevance:** decision rule = run iff `observed_kurtosis > KURTOSIS_TRIGGER` (3.0). Z-2-W shares all other pins with Z-2-main (block_len=4, B_PATHS=1,000, no re-anchoring) and adds SEED_Z2_W=20260521.
- **Connection:** Z-3 escalation evaluates BOTH Z-2-main AND Z-2-W; if only Z-2-main exceeds and Z-2-W does not, the trigger is heavy-tail-driven (diagnostic distinction).

### Why

If `observed_kurtosis > 3.0`, build the Winsorized pool `clip(dln_usd, q05, q95)` and re-run the backcast bootstrap. Otherwise, skip with an explicit reason.

In [8]:
if observed_kurtosis > KURTOSIS_TRIGGER:
    lo_q, hi_q = float(np.quantile(dln_usd, 0.05)), float(np.quantile(dln_usd, 0.95))
    dln_usd_w = np.clip(dln_usd, lo_q, hi_q)
    n_clipped_lo = int((dln_usd < lo_q).sum())
    n_clipped_hi = int((dln_usd > hi_q).sum())
    print(f'Z-2-W ACTIVATED: observed_kurtosis={observed_kurtosis:.4f} > {KURTOSIS_TRIGGER}')
    print(f'  Winsorize bounds: q05={lo_q:+.5f}, q95={hi_q:+.5f}')
    print(f'  Clipped: {n_clipped_lo} below q05, {n_clipped_hi} above q95 (of {T_daily})')

    fx_shares_w = backcast_fx_shares(
        pool_dln_usd=dln_usd_w,
        trm_arr=trm_arr_backcast,
        cost_start_usd=COST_START_USD,
        n_paths=B_PATHS,
        block_len=BLOCK_LEN_BACKCAST,
        seed=SEED_Z2_W,
    )
    median_fx_w = float(np.median(fx_shares_w))
    ci_w_lo, ci_w_hi = float(np.quantile(fx_shares_w, 0.05)), float(np.quantile(fx_shares_w, 0.95))
    print(f'\nZ-2-W per-path FX share distribution:')
    print(f'  median             = {median_fx_w:.7f}')
    print(f'  90% inter-path CI  = [{ci_w_lo:.7f}, {ci_w_hi:.7f}]')
    print(f'  Z-2-W median / daily baseline = {median_fx_w / DAILY_BASELINE_FX_SHARE:.2f}x')
else:
    median_fx_w = None
    ci_w_lo = ci_w_hi = float('nan')
    print(f'Z-2-W SKIPPED: observed_kurtosis={observed_kurtosis:.4f} <= {KURTOSIS_TRIGGER}')
    print('  Heavy-tail diagnostic not required by spec under this kurtosis.')

Z-2-W ACTIVATED: observed_kurtosis=3.6237 > 3.0
  Winsorize bounds: q05=-2.56219, q95=+2.11112
  Clipped: 2 below q05, 2 above q95 (of 28)



Z-2-W per-path FX share distribution:
  median             = 0.0000333
  90% inter-path CI  = [0.0000310, 0.0000358]
  Z-2-W median / daily baseline = 1.20x


### Interpretation (Z-2-W)

If Z-2-W ran: comparison of Z-2-W median to Z-2-main median diagnoses heavy-tail influence. If they agree, the small-FX finding is not heavy-tail-driven. If they diverge, the disposition memo records the distinction before any escalation. If Z-2-W was skipped, the kurtosis-trigger gate did not fire.

## Trio Z-3 — Escalation gate + close-out routing

### Decision-citation block (4-part)

- **Reference:** spec §0.6 CORRECTIONS-Z-3 + RC Q6 compound close-out gate.
- **Why:** R6 sibling-iteration activation costs 1-2 weeks of implementation; the gate must be pre-pinned, numeric, and respect the null-calibration suppressor so we don't pay that cost on a false alarm.
- **Relevance:** evaluates OR(rel ≥ 5×baseline, abs ≥ 0.05) on each of Z-2-main and Z-2-W; if EITHER triggers AND null calibration passes, escalate. Monthly stratum is EXCLUDED (RC Q3 pin); weekly is reported but interpreted under the ±2× rule, not the 5× rule.
- **Connection:** routes the iteration to one of three states — ESCALATE (R6 activation), NOT ESCALATE (clean Z-2 corroboration), or SUPPRESSED (null calibration tripped). Then the compound close-out gate combines Z-3 with Y-9 ccusage-parity-0.1% status.

### Why

Evaluate the escalation predicate per the pre-pinned rule and route the iteration.

In [9]:
rel_threshold = ESCALATION_REL * DAILY_BASELINE_FX_SHARE

def trips(x):
    if x is None:
        return False
    return bool((x >= rel_threshold) or (x >= ESCALATION_ABS))

main_trips = trips(median_fx_main)
w_trips    = trips(median_fx_w)
raw_escalate = main_trips or w_trips

if NULL_GATE_TRIPPED:
    Z3_VERDICT = 'SUPPRESSED'
    Z3_REASON  = (f'Z-2-null median {median_fx_null:.7f} itself trips the escalation gate '
                  f'(>= {rel_threshold:.7f}); bootstrap mechanics cannot distinguish signal from MC '
                  f'noise. R6 NOT activated; diagnostic memo for future iteration.')
elif raw_escalate:
    Z3_VERDICT = 'ESCALATE'
    reasons = []
    if main_trips:
        reasons.append(f'Z-2-main median {median_fx_main:.7f} trips')
    if w_trips:
        reasons.append(f'Z-2-W median {median_fx_w:.7f} trips')
    Z3_REASON = '; '.join(reasons) + f' (threshold rel={rel_threshold:.7f} OR abs={ESCALATION_ABS}).'
else:
    Z3_VERDICT = 'NOT_ESCALATE'
    pieces = [f'Z-2-main {median_fx_main:.7f}']
    if median_fx_w is not None:
        pieces.append(f'Z-2-W {median_fx_w:.7f}')
    Z3_REASON = ('All Z-2 arms below escalation thresholds: ' + ', '.join(pieces)
                 + f' vs rel={rel_threshold:.7f}, abs={ESCALATION_ABS}.')

print('=' * 78)
print('Z-3 escalation gate evaluation')
print('=' * 78)
print(f'Pre-pinned thresholds: rel = {ESCALATION_REL}x baseline = {rel_threshold:.7f}; abs = {ESCALATION_ABS}')
print(f'Null-calibration status: NULL_GATE_TRIPPED = {NULL_GATE_TRIPPED}')
print(f'\nZ-2-main median = {median_fx_main:.7f}  trips? {main_trips}')
if median_fx_w is not None:
    print(f'Z-2-W  median   = {median_fx_w:.7f}  trips? {w_trips}')
else:
    print('Z-2-W  median   = SKIPPED (kurtosis below trigger)')
print(f'\nZ-3 VERDICT: {Z3_VERDICT}')
print(f'Reason:      {Z3_REASON}')

if Z3_VERDICT == 'ESCALATE':
    print('\n>>> Route: ACTIVATE R6 sibling iteration')
    print('    docs/specs/2026-05-16-r6-continuous-stream-simulation-design.md')
    print('    docs/plans/2026-05-16-r6-continuous-stream-simulation-plan.md')
elif Z3_VERDICT == 'NOT_ESCALATE':
    print('\n>>> Route: small-FX finding is REGIME-CORROBORATED at the 2024-2025 TRM window.')
    print('    R6 escalation NOT triggered. Compound close-out gate evaluated below.')
else:
    print('\n>>> Route: gate SUPPRESSED — record diagnostic memo; do not escalate or close out on Z-3.')

Z-3 escalation gate evaluation
Pre-pinned thresholds: rel = 5.0x baseline = 0.0001385; abs = 0.05
Null-calibration status: NULL_GATE_TRIPPED = False

Z-2-main median = 0.0000194  trips? False
Z-2-W  median   = 0.0000333  trips? False

Z-3 VERDICT: NOT_ESCALATE
Reason:      All Z-2 arms below escalation thresholds: Z-2-main 0.0000194, Z-2-W 0.0000333 vs rel=0.0001385, abs=0.05.

>>> Route: small-FX finding is REGIME-CORROBORATED at the 2024-2025 TRM window.
    R6 escalation NOT triggered. Compound close-out gate evaluated below.


### Interpretation (Z-3)

Z-3 routes the iteration to one of three states (ESCALATE / NOT_ESCALATE / SUPPRESSED). The verdict is pre-determined by the pinned thresholds — implementers do not retune. The compound close-out gate (next cell) combines this Z-3 verdict with Y-9 status.

## Headline output + compound close-out gate (Y-9 + Z-3)

Final tabulation per the implementer plan. The compound close-out gate (CR Q6 pin) requires BOTH Z-3 non-trigger AND Y-9 ccusage-parity-0.1% closure. Y-9 is NOT closed as of v0.2.6 → expected disposition is PAUSED.

In [10]:
# Y-9 status is a spec-level fact, not derived in this notebook.
# Per v0.2.6 §0.5 Y-8 parity-target-status pin (CR NIT-4): Y-9 ccusage-parity-0.1% closure
# is a v0.2.7+ work item; it is NOT closed at v0.2.6.
Y9_CLOSED = False
Y9_NOTE = ('Y-9 ccusage-parity-0.1% closure is a v0.2.7+ work item; current panel parity is '
           '0.9994x (2.3% over ccusage after Y-8 fix), still outside the 0.1% target.')

if Z3_VERDICT == 'ESCALATE':
    closeout = 'PAUSED (Z-3 routes to R6 activation; not closing v0.2.6 here)'
elif Z3_VERDICT == 'SUPPRESSED':
    closeout = 'PAUSED (Z-3 suppressed by null-calibration; cannot close on a mis-calibrated gate)'
elif Z3_VERDICT == 'NOT_ESCALATE' and Y9_CLOSED:
    closeout = 'CLOSED at v0.2.6'
else:
    closeout = 'PAUSED (Z-3 non-trigger but Y-9 not yet closed; compound gate requires BOTH)'

print('=' * 78)
print('HEADLINE — v0.2.6 §0.6 CORRECTIONS-Z sensitivity arms')
print('=' * 78)
print(f'Daily baseline FX share (Task 12 R5, pinned):     {DAILY_BASELINE_FX_SHARE:.7f}')
print(f'observed_kurtosis (Fisher excess, unbiased):       {observed_kurtosis:.4f}  '
      f'({"Z-2-W ran" if observed_kurtosis > KURTOSIS_TRIGGER else "Z-2-W skipped"})')
print()
print('--- Z-1 multi-period aggregation ---')
print(f'  Z-1a weekly   FX share = {pt_fx_weekly:.6f}   90% CI = [{fx_wk_lo:.6f}, {fx_wk_hi:.6f}]')
print(f'  Z-1b monthly  FX share = {pt_fx_monthly:.6f}   90% CI = [{fx_mo_lo:.6f}, {fx_mo_hi:.6f}]')
print(f'                          TAG: uninformative — T_dln={T_mo_dln} (EXCLUDED from Z-3)')
print()
print('--- Z-2 backcast bootstrap ---')
print(f'  Z-2-main    median = {median_fx_main:.7f}   90% inter-path = [{ci_main_lo:.7f}, {ci_main_hi:.7f}]')
print(f'  Z-2-null    median = {median_fx_null:.7f}   90% inter-path = [{ci_null_lo:.7f}, {ci_null_hi:.7f}]')
if median_fx_w is not None:
    print(f'  Z-2-W       median = {median_fx_w:.7f}   90% inter-path = [{ci_w_lo:.7f}, {ci_w_hi:.7f}]')
else:
    print(f'  Z-2-W       SKIPPED (kurtosis {observed_kurtosis:.4f} <= {KURTOSIS_TRIGGER})')
print()
print('--- Z-3 verdict ---')
print(f'  VERDICT: {Z3_VERDICT}')
print(f'  REASON:  {Z3_REASON}')
print()
print('--- Compound close-out gate (CR Q6 pin: Z-3 AND Y-9) ---')
print(f'  Z-3 status:  {Z3_VERDICT}')
print(f'  Y-9 closed:  {Y9_CLOSED}')
print(f'  Y-9 note:    {Y9_NOTE}')
print(f'  Disposition: {closeout}')
print()
print('  v0.2.6 close-out requires BOTH Z-3 non-trigger AND Y-9 ccusage-parity-0.1% closure.')
print('  Y-9 not yet closed -> iteration PAUSED at v0.2.6, not closed.')

HEADLINE — v0.2.6 §0.6 CORRECTIONS-Z sensitivity arms
Daily baseline FX share (Task 12 R5, pinned):     0.0000277
observed_kurtosis (Fisher excess, unbiased):       3.6237  (Z-2-W ran)

--- Z-1 multi-period aggregation ---
  Z-1a weekly   FX share = 0.000073   90% CI = [-0.000290, 0.000130]
  Z-1b monthly  FX share = 0.000099   90% CI = [nan, nan]
                          TAG: uninformative — T_dln=3 (EXCLUDED from Z-3)

--- Z-2 backcast bootstrap ---
  Z-2-main    median = 0.0000194   90% inter-path = [0.0000169, 0.0000223]
  Z-2-null    median = 0.0000293   90% inter-path = [0.0000182, 0.0000581]
  Z-2-W       median = 0.0000333   90% inter-path = [0.0000310, 0.0000358]

--- Z-3 verdict ---
  VERDICT: NOT_ESCALATE
  REASON:  All Z-2 arms below escalation thresholds: Z-2-main 0.0000194, Z-2-W 0.0000333 vs rel=0.0001385, abs=0.05.

--- Compound close-out gate (CR Q6 pin: Z-3 AND Y-9) ---
  Z-3 status:  NOT_ESCALATE
  Y-9 closed:  False
  Y-9 note:    Y-9 ccusage-parity-0.1% closure is